# **Анализ выживаемости на Титанике**

## 1. Загрузка данных и их анализ

Данный учебный проект основан на знаменитом датасете "Титаник".

**Цель проекта** - выяснить, какие факторы влияли на выживаемость 
людей при кораблекрушении.

При знакомстве с датасетом после загрузки была выведена основная 
информация: количество и названия столбцов, первые 5 строк, shape 
таблицы, типы данных и количество непустых значений. 
Также создана копия датафрейма для сохранности исходных данных.

**Датасет:** 891 пассажир, 12 признаков.

In [1]:
import pandas as pd

data = pd.read_csv('train.csv')
data_copy = data.copy()
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
print(data.shape)
print(data.info())

(891, 12)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None


## 2. Очистка данных

При первичном осмотре с помощью `data.info()` обнаружены пропуски:
- **Age** - 177 пропусков
- **Cabin** - 687 пропусков  
- **Embarked** - 2 пропуска

Каждый столбец обработан исходя из того, какой метод наименее исказит анализ:

- **Age** заполнен медианой - она точнее среднего при наличии выбросов
- **Cabin** удалён - 687 из 891 строк пустые (77%), столбец непригоден для анализа
- **Embarked** заполнен наиболее частым значением (S - Southampton)

После каждого заполнения выполнялась проверка через `isnull().sum()`.

In [3]:
data.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
data['Age'] = data['Age'].fillna(data['Age'].median())
print(data['Age'].isnull().sum())

0


In [5]:
data = data.drop(columns = ['Cabin'])
data.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       2
dtype: int64

In [6]:
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
print(data['Embarked'].isnull().sum())

0


## 3. Выживаемость по полу и классу

Анализ показал, что выживаемость среди женщин кратно выше чем 
среди мужчин во всех классах:
- Женщины 1 класса: **96.8%**
- Женщины 2 класса: **92.1%**  
- Женщины 3 класса: **50.0%**
- Мужчины 1 класса: **36.9%**
- Мужчины 2 класса: **15.7%**
- Мужчины 3 класса: **13.5%**

При кораблекрушении был отдан приказ спасать женщин и детей  в первую очередь. Разница по классам объясняется расположением кают: 1 класс находился на верхних палубах ближе к шлюпкам, пассажиры 3 класса физически не успевали добраться до них.

In [7]:
data.groupby(['Pclass', 'Sex'])['Survived'].mean()*100

Pclass  Sex   
1       female    96.808511
        male      36.885246
2       female    92.105263
        male      15.740741
3       female    50.000000
        male      13.544669
Name: Survived, dtype: float64

## 4. Средний возраст по полу и классу

- Женщины 1 класса: **32.5 лет** | Мужчины 1 класса: **36.0 лет**
- Женщины 2 класса: **28.0 лет** | Мужчины 2 класса: **29.0 лет**
- Женщины 3 класса: **28.0 лет** | Мужчины 3 класса: **28.0 лет**

Средний возраст женщин и мужчин в каждом классе примерно одинаковый. В 1 и 2 классах мужчины статистически старше женщин на 1-3,5 года.

Основываясь на социальных тенденциях начала 20 века можно предположить, что на корабле преобладали семьи, где женщина была ровесницей мужа или младше него - что отражает характерную для той эпохи разницу в возрасте между супругами.

In [8]:
data.groupby(['Pclass', 'Sex'])['Age'].median()

Pclass  Sex   
1       female    32.5
        male      36.0
2       female    28.0
        male      29.0
3       female    28.0
        male      28.0
Name: Age, dtype: float64

## 5. Топ-5 самых молодых выживших

Все пятеро самых молодых выживших - дети до года из 2 и 3 классов. Это подтверждает что при эвакуации дети имели приоритет вне зависимости от класса каюты.

In [9]:
survived_name_age = data[data['Survived'] == 1]
survived_name_age.nsmallest(5, 'Age')[['Name', 'Age', 'Pclass']]

,Name,Age,Pclass
803,"Thomas, Master. Assad Alexander",0.42,3
755,"Hamalainen, Master. Viljo",0.67,2
469,"Baclini, Miss. Helene Barbara",0.75,3
644,"Baclini, Miss. Eugenie",0.75,3
78,"Caldwell, Master. Alden Gates",0.83,2


## 6. Медианная цена билета по классу и полу

Одно из самых интересных наблюдений: медианная цена билета женщин из любого класса примерно в 2 раза больше медианной цены билета для мужчин. Это связано с тем, что, во-первых, женщины путешествовали с состоятельными мужьями, поэтому класс выше - и цена выше, а во-вторых, на кораблях того времени женщины размещались в отдельных каютах, а не в общих помещениях - каюта дороже.

In [10]:
data.groupby(['Pclass', 'Sex'])['Fare'].median()

Pclass  Sex   
1       female    82.66455
        male      41.26250
2       female    22.00000
        male      13.00000
3       female    12.47500
        male       7.92500
Name: Fare, dtype: float64

## 7. Общие выводы

На Титанике преобладали семьи. При кораблекрушении в первую очередь были спасены женщины и дети - это подтверждается данными: наибольшая выживаемость среди женщин 1 и 2 классов, а самые младшие выжившие имели места во 2 и 3 классах.

Для первого проекта были выбраны признаки выживаемости, возраста и цены билетов - как наиболее привлекательные для дальнейшего развития анализа этого датасета в будущем.